In [3]:
import sqlite3

In [4]:
from datetime import datetime

In [ ]:
# Initialization
def get_connection():
    return sqlite3.connect("telehealth_1.db")

def init_db():
    conn = get_connection()
    conn.execute("PRAGMA foreign_keys = ON") 
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS patients (
                    patient_id INTEGER PRIMARY KEY,
                    first_name TEXT NOT NULL,
                    last_name TEXT NOT NULL,
                    age INTEGER,
                    phone TEXT NOT NULL,
                    email TEXT UNIQUE)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS appointments (
                    appointment_id INTEGER PRIMARY KEY AUTOINCREMENT,
                    patient_id INTEGER,
                    date_time TEXT,
                    status TEXT,
                    FOREIGN KEY(patient_id) REFERENCES patients(patient_id)
                    ON DELETE RESTRICT
                    ON UPDATE CASCADE )''') 
    conn.commit()
    conn.close()

init_db()


In [6]:
class Patient:
    def __init__(self, patient_id, first_name, last_name, age, phone, email):
        self.patient_id = patient_id
        self.first_name = first_name
        self.last_name = last_name
        self.age = age
        self.phone = phone
        self.email = email

    def save_to_db(self):
        with sqlite3.connect("telehealth_1.db") as conn:
            conn = get_connection()
            cursor = conn.cursor()

            cursor.execute("""
            INSERT INTO patients 
            (patient_id, first_name, last_name, age, phone, email)
            VALUES (?, ?, ?, ?, ?, ?)
            """, (
                self.patient_id,
                self.first_name,
                self.last_name,
                self.age,
                self.phone,
                self.email
            ))

            conn.commit()
            conn.close()

#get a list of all patients    
    @staticmethod
    def list_all():
        with sqlite3.connect("telehealth_1.db") as conn:
            conn = get_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM patients")
            rows = cursor.fetchall()
            conn.close()
        return rows
    
#get a specific patient 
    @staticmethod
    def get_by_id(patient_id):
        with sqlite3.connect("telehealth_1.db") as conn:
            conn = get_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM patients WHERE patient_id = ?", (patient_id,))
            row = cursor.fetchone()
            conn.close()
        return row


In [7]:
class Appointment:
    def __init__(self, appointment_id, patient_id, date_time, status="Scheduled"):
        self.appointment_id = appointment_id
        self.patient_id = patient_id
        self.date_time = date_time
        self.status = status

    def save_to_db(self):
        with sqlite3.connect("telehealth_1.db") as conn:
            conn = get_connection()
            cursor = conn.cursor()

            cursor.execute("""
            INSERT INTO appointments 
            (appointment_id, patient_id, date_time, status)
            VALUES (?, ?, ?, ?)
            """, (
                self.appointment_id,
                self.patient_id,
                self.date_time,
                self.status
            ))

            conn.commit()
            conn.close()
        
#get an appointement for a specific patient
    @staticmethod
    def for_patient(patient_id):
        with sqlite3.connect("telehealth_1.db") as conn:
            conn = get_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT * FROM appointments WHERE patient_id = ?", (patient_id,))
            rows = cursor.fetchall()
            conn.close()
        return rows

In [8]:
class Triage:
    SYMPTOM_RULES = {
        "fever": 2,
        "cough": 1,
        "chest pain": 3,
        "shortness of breath": 3,
        "headache": 12
    }

    @staticmethod
    def assess_symptoms(symptoms):
        severity = sum(Triage.SYMPTOM_RULES.get(s.lower(), 0) for s in symptoms)
        if severity >= 3:
            return "High"
        elif severity == 2:
            return "Moderate"
        return "Low"


class NotificationService:
    @staticmethod
    def send_reminder(patient, appointment):
        print(f"Reminder: {patient.first_name} {patient.last_name}, "
              f"appointment at {appointment.date_time}")

In [ ]:
import csv

with open("patients.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        p = Patient(
            int(row["patient_id"]),
            row["first_name"],
            row["last_name"],
            int(row["age"]),
            row["phone"],
            row["email"]
        )
        p.save_to_db()


with open("appointments.csv") as file:
    reader = csv.DictReader(file)
    for row in reader:
        a = Appointment(
            int(row["appointment_id"]),
            int(row["patient_id"]),
            row["date_time"],
            row["status"]
        )
        a.save_to_db()

print("CSV data loaded into the database successfully!")


CSV data loaded into the database successfully!


In [ ]:
 # Interactive Telehealth Menu
while True:
    print("\n Telehealth Management Menu ")
    print("1. Register New Patient")
    print("2. Schedule Appointment")
    print("3. View Appointments & Send Reminders")
    print("4. Run Symptom Triage")
    print("5. Exit")

    choice = input("Enter your choice (1-5): ").strip()


    if choice == "1":   # Register New Patient 

        first_name = input("First Name: ")
        last_name = input("Last Name: ")
        age = int(input("Age: "))
        phone = input("Phone: ")
        email = input("Email: ")

        existing_patients = Patient.list_all()
        existing_ids = [row[0] for row in existing_patients]
        patient_id = max(existing_ids + [0]) + 1

        new_patient = Patient(patient_id, first_name, last_name, age, phone, email)
        new_patient.save_to_db()

        print(f"Patient {first_name} {last_name} registered with ID {patient_id}")


    elif choice == "2":   # Schedule Appointment 

        patients = Patient.list_all()

        print("Available Patients:")
        for row in patients:
            print(f"{row[0]}: {row[1]} {row[2]}")

        patient_id = int(input("Enter Patient ID to schedule for: "))
        date_time = input("Appointment Date & Time (YYYY-MM-DD HH:MM): ")

        # Generate next appointment ID and save
        all_appointments = []
        for row in patients:
            all_appointments.extend(Appointment.for_patient(row[0]))

        existing_ids = [appt[0] for appt in all_appointments]
        appointment_id = max(existing_ids + [0]) + 1

        new_appointment = Appointment(appointment_id, patient_id, date_time)
        new_appointment.save_to_db()

        print(f"Appointment for Patient ID {patient_id} scheduled at {date_time}")


    elif choice == "3":   # View Appointments & Reminders 

        patients = Patient.list_all()

        for row in patients:
            current_patient = Patient(*row)

            appointments = Appointment.for_patient(current_patient.patient_id)

            for appt in appointments:
                current_appointment = Appointment(*appt)

                print(
                    f"{current_patient.first_name} {current_patient.last_name} "
                    f"- {current_appointment.date_time} - {current_appointment.status}"
                )

                send = input(f"Send reminder to {current_patient.first_name}? (y/n): ").lower()

                if send == "y":
                    NotificationService.send_reminder(current_patient, current_appointment)


    elif choice == "4":   # Symptom Triage

        patients = Patient.list_all()

        print("Available Patients:")
        for row in patients:
            print(f"{row[0]}: {row[1]} {row[2]}")

        patient_id = int(input("Enter Patient ID for triage: "))

        selected_row = [row for row in patients if row[0] == patient_id][0]
        selected_patient = Patient(*selected_row)

        symptoms_input = input("Enter symptoms separated by commas: ")
        symptoms = [s.strip() for s in symptoms_input.split(",")]

        severity = Triage.assess_symptoms(symptoms)

        print(
            f"Patient {selected_patient.first_name} {selected_patient.last_name} "
            f"- Triage Severity: {severity}"
        )


    elif choice == "5":
        print("Exiting Telehealth Management Menu. Goodbye!")
        break


    else:
        print("Invalid choice. Please enter a number 1-5.")




=== Telehealth Management Menu ===
1. Register New Patient
2. Schedule Appointment
3. View Appointments & Send Reminders
4. Run Symptom Triage
5. Exit


Patient Bridget MAina registered with ID 6

=== Telehealth Management Menu ===
1. Register New Patient
2. Schedule Appointment
3. View Appointments & Send Reminders
4. Run Symptom Triage
5. Exit
Available Patients:
1: Alice Maina
2: Brian Kariuki
3: Clara Wanjiku
4: David Otieno
5: Esther Muthoni
6: Bridget MAina
Appointment for Patient ID 6 scheduled at 2026-02-10 10:30

=== Telehealth Management Menu ===
1. Register New Patient
2. Schedule Appointment
3. View Appointments & Send Reminders
4. Run Symptom Triage
5. Exit
Alice Maina - 2026-03-10 10:30 - Scheduled
Reminder: Alice Maina, appointment at 2026-03-10 10:30
Alice Maina - 2026-03-12 14:00 - Completed
Brian Kariuki - 2026-03-10 11:00 - Scheduled
Clara Wanjiku - 2026-03-11 09:00 - Scheduled
Reminder: Clara Wanjiku, appointment at 2026-03-11 09:00
David Otieno - 2026-03-13 16:00 - Missed
Reminder: David Otieno, appointment at 2026-03-13 16:00
Bridget MAina - 2026-02-10 10:30 - Scheduled
Reminder: Bridget MAina, appointment at 2026